# 07 — Haiku judge for quantized deployment conditions

Select the **local** `.venv-local` kernel. The same frozen six-dimension common rubric from notebook 03 judges one quantized candidate per independent API request, with the corresponding Haiku reference and ground truth. Only records with `status=completed` are semantic-judge candidates, including completed empty, malformed, truncated or otherwise poor responses. Missing, `context_unsupported`, and retryable/error records have no returned candidate response, so they are omitted from paid judging but retained in coverage and are never assigned synthetic zero scores. The static rubric alone is marked for prompt caching; candidate cases are never cached or combined. Cache hits are opportunistic in batches and actual cache usage is retained per response. Condition C is optional and uses a separate ledger so it can be run later.

## 1. Local environment and frozen source

Read source, code and interpreter identity. No tunnel or local receiver is needed for local Haiku API calls.

**Recovery / safe rerun:** After local kernel restart rerun 1, then the interrupted section.

**Cost:** No paid API requests.



In [1]:
from pathlib import Path
import sys
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p/'streamlined/common.py').is_file()), None)
if ROOT is None:
    ROOT = next(p/'streamlined_experiment' for p in candidates if (p/'streamlined_experiment/streamlined/common.py').is_file())
sys.path.insert(0, str(ROOT))
from streamlined.common import *
print('Experiment:', ROOT)
print('Interpreter:', sys.executable)

from streamlined.common import verify_experiment
print('Frozen:',verify_experiment(ROOT)['config_hash'])

Experiment: d:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment
Interpreter: d:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment\.venv-local\Scripts\python.exe
Frozen: 49e13df7571913d2c39067dfd7dcde93353110b2f4c7523bc8a18de2459b4b73


## 2. Confirm API access and planned request scope

Load `secret/Claude_API_Key.txt` from the project tree. The key is held in memory and never printed. Section 2 inventories every expected row, but only completed records enter the paid judge request set. Missing, `context_unsupported`, retryable-error and other non-completed rows are reported as `omitted_no_response` and do not interrupt freezing. This loses no generated response because those statuses have no returned `response_raw`; a successfully returned empty response is `completed` and remains included.

**Recovery / safe rerun:** Rerun 1–2 after a kernel restart. Finish additional 04/05 rows before first freezing Section 3 if they are to enter this ledger; once Section 3 freezes paid request identities, later candidate additions require a separately named judge protocol rather than silently changing this run.

**Cost:** Model metadata lookup only; no judge requests.



In [2]:
from streamlined.api import client
from continuation.judge import source_rows
api=client(ROOT)
print('Model:',api.models.retrieve('claude-haiku-5-5').id)
required_rows,coverage=source_rows(ROOT,'required')
print('Expected candidate rows:',sum(int(row['expected']) for row in coverage))
print('Paid judge requests planned (completed responses):',len(required_rows))
print('Omitted rows with no returned response:',sum(int(row['omitted_no_response']) for row in coverage))
print('Required streams:',len(coverage))
print('Exact per-stream status counts are written to source_coverage.csv when Section 3 freezes the ledger.')

Model: claude-haiku-5-5
Expected candidate rows: 12780
Paid judge requests planned (completed responses): 12212
Omitted rows with no returned response: 568
Required streams: 30
Exact per-stream status counts are written to source_coverage.csv when Section 3 freezes the ledger.


## 3. Freeze common cached judge jobs

Construct identical one-row judge cases with the notebook 03 rubric for every available completed response. Only the static system rubric gets a cache-control marker. The first run freezes model, rubric, prompt and candidate identity in the request manifest; recovery reruns reload that exact frozen job dictionary and paid ledger even if source inference files subsequently change. Non-completed rows remain documented in the frozen `source_coverage.csv` and are not fabricated as candidate answers.

**Recovery / safe rerun:** Rerun 1–3 after a kernel restart. If source candidates change, finish their own run and use a new judge protocol name; this ledger refuses changes.

**Cost:** No paid judge requests.



In [3]:
from continuation.judge import make_run
required=make_run(ROOT,'required')
print(required.counts())
print('Cache policy:',required.folder/'cache_policy.json')

{'total': 12212, 'completed': 1455, 'submitted': 10757}
Cache policy: d:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment\results\api\quantized_common_cached_v1\cache_policy.json


## 4. Paid five-row pilot

Inspect five real judgments, raw output and token/cache usage. All pilot rows are checkpointed and skipped by the main batch.

**Recovery / safe rerun:** Rerun 1–3 then 4 only if the pilot is incomplete. Unknown outcomes remain held; inspect receipts and resolve them before an explicit retry.

**Cost:** Up to five paid Messages API requests.



In [ ]:
print(required.pilot(size=5,retry_failed=True))
print(required.export())

## 5. Reconcile ambiguous batch submissions

Find a previously submitted remote batch by exact custom-ID set; this step never creates a new paid request.

**Recovery / safe rerun:** Rerun 1–3 then 5 after a network interruption. Repeat until unknown submissions are resolved.

**Cost:** Read-only API listing and result retrieval.



In [ ]:
print(required.reconcile())
print(required.export())

## 6. Submit remaining required candidates

Batch one candidate per API request. Submitted IDs are held in the local ledger, so rerunning does not duplicate them. Rate-limit rejections can be retried after waiting; unknown outcomes must be reconciled first.

**Recovery / safe rerun:** Rerun 1–3, 5, then 6. Do not erase the submissions ledger; a 429 may need a cooldown.

**Cost:** Paid Message Batches API for remaining required candidates; prompt-cache hits are not guaranteed.



In [ ]:
print(required.submit(batch_size=50,retry_failed=True))
print(required.export())

## 7. Poll and collect required results

Collect ended batches. Polling is paced below the organisation's Message Batches API request-rate limit and automatically pauses then resumes after HTTP 429 responses; this changes neither requests nor judgments. Every returned raw judgment and usage object is saved as an atomic local JSON record. The CSV is refreshed every 100 record mutations and at the end of this cell; each record is saved immediately.

**Recovery / safe rerun:** Rerun 1–3 then 7 until no pending batch remains. Polling never submits new paid work.

**Cost:** Read-only batch polling and result collection.



In [4]:
print(required.poll(max_seconds=3600))
print(required.export())

{'total': 12212, 'completed': 12212}
{'total': 12212, 'completed': 12212}
{'total': 12212, 'completed': 12212}


## 8. Optional IQ1_S: freeze jobs

Run after all five optional IQ1_S untreated and 20 adapter streams finish. This separate ledger preserves the same rubric/cache design and never alters required judgments.

**Recovery / safe rerun:** Rerun 1 and 8 after a kernel restart. If IQ1_S is not being performed, skip sections 8–11.

**Cost:** No paid judge requests.



In [ ]:
optional=make_run(ROOT,'optional')
print(optional.counts())

## 9. Optional IQ1_S: five-row pilot

Verify one-row judgments and actual cache usage before optional batch submission.

**Recovery / safe rerun:** Rerun 1 and 8–9 as needed; completed pilot rows are skipped.

**Cost:** Up to five paid Messages API requests.



In [ ]:
print(optional.pilot(size=5,retry_failed=True))
print(optional.export())

## 10. Optional IQ1_S: reconcile and submit

Resolve unknown batch outcomes before submitting optional judgments.

**Recovery / safe rerun:** Rerun 1, 8, then 10. Unknown submissions remain held to avoid duplicate charges.

**Cost:** Paid Message Batches API for remaining optional candidates.



In [ ]:
print(optional.reconcile())
print(optional.submit(batch_size=50,retry_failed=True))
print(optional.export())

## 11. Optional IQ1_S: poll results

Collect, checkpoint and export optional judgments.

**Recovery / safe rerun:** Rerun 1, 8, then 11 until pending reaches zero; polling does not submit.

**Cost:** Read-only batch polling and result collection.



In [ ]:
print(optional.poll(max_seconds=3600))
print(optional.export())